# Movie Crawler Website

In [3]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import json

## 1. Extraíndo filmes da página atual

In [4]:
#Objetos
user_agent = 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Mobile Safari/537.36'
headers = {'User-Agent': user_agent}
#links
url = 'https://subslikescript.com'
listagem = f'{url}/movies'

# Coletando site
html_site = requests.get(listagem, headers = headers)
saida = BeautifulSoup(html_site.text, 'html.parser')

# Extraíndo links das páginas:
caixa =  saida.find('ul', class_ = 'scripts-list')
hiperlinks = caixa.find_all('a', href = True)

#Gravando path completo:
links = [url + a['href'] for a in hiperlinks]

#Inicializando dict
dataframe = pd.DataFrame(columns = ['Title', 'Script'])

#Scrapping sites
for url_hiperlink in links:
    # Obtendo html de cada site
    try:
        result  = requests.get(url = url_hiperlink, headers = headers)
        result = result.text
        
    except Exception as e:
        print(f'Falha na coleta em \n{url_hiperlink}')
        print(f'Erro: {e}')
    else:
        soup = BeautifulSoup(result, 'lxml')
        title = soup.find('article', class_ = 'main-article').find('h1').get_text(strip = True)
        script = soup.find('div', class_ = 'full-script').get_text(strip=True, separator = ' ')
        temp_df = {
                'Title': title.split(' -')[0],
                'Script': script
            }
        df = pd.DataFrame([temp_df])
        dataframe = pd.concat([dataframe, df], axis = 0,  ignore_index = True)

## 2. Com Paginação

In [5]:
#links
letra = 'aux variable'
while len(letra) != 1:
    letra = input('Escolha a letra inicial dos filmes (A-Z) que quer coletar:\n')

url = 'https://subslikescript.com'
listagem = f'{url}/movies_letter-{letra.upper()}'

# Coletando site
html_site = requests.get(listagem, headers = headers)
saida = BeautifulSoup(html_site.text, 'html.parser')

# ult_pag = saida.find('ul', class_ = 'pagination').find_all('li', class_ = 'page-item')[-2]
# ult_pag = int(ult_pag.text)

#Inicializando df
dataframe_paginado = pd.DataFrame(columns = ['Title', 'Script'])
    
#for page in range(1, ult_pag+1,1):
for page in range(1, 3,1):

    pagina = f'{listagem}/?page={page}'
    html_site = requests.get(pagina, headers = headers)
    saida = BeautifulSoup(html_site.text, 'html.parser')
    
    # Extraíndo links das páginas:
    caixa = saida.find('ul', class_ = 'scripts-list')
    hiperlinks = caixa.find_all('a', href = True)
    
    #Gravando path completo:
    links = [url + a['href'] for a in hiperlinks]
    
    #Scrapping sites
    for url_hiperlink in links:
        # Obtendo html de cada site
        try:
            result  = requests.get(url = url_hiperlink, headers = headers)
            result = result.text
            
        except Exception as e:
            print(f'Falha na coleta em \n{url_hiperlink}')
            print(f'Erro: {e}')
        else:
            soup = BeautifulSoup(result, 'lxml')
            title = soup.find('article', class_ = 'main-article').find('h1').get_text(strip = True)
            script = soup.find('div', class_ = 'full-script').get_text(strip=True, separator = ' ')
            temp_df = {
                'Title': title.split(' -')[0],
                'Script': script
            }
            df = pd.DataFrame([temp_df])
            dataframe_paginado = pd.concat([dataframe, df], axis = 0,  ignore_index = True)



In [6]:
dataframe_paginado.head(5)

,Title,Script
0,Nasty (2022),(electronic tapping) (upbeat music) (upbeat mu...
1,Kathleen Madigan: Hunting Bigfoot (2023),[UPBEAT MUSIC PLAYING] [CROWD CHEERING] Wow. H...
2,Cassius X: Becoming Ali (2023),ELIJAH MUHAMMAD: Cassius Clay. He was box offi...
3,Verden er uskarp (2022),The almost 2.5 years I worked at the Bauhaus s...
4,Bullet to Beijing (1995),"Morning, Carruthers. You're late. I waited a h..."
